# Hélio — pipeline Datajud integrado ao projeto

Este notebook coleta os metadados da API pública do Datajud, transforma cada resposta para o contrato do modelo `DatajudRegistro` e salva a tabela consolidada em Parquet.

O arquivo histórico será salvo em `data/processed/datajud/tabela_principal_helios_2000_2024.parquet`, sem sobrescrever o arquivo existente de 2025–2026. O comando posterior de importação usará os dois arquivos. A chave da API nunca fica no notebook: use `DATAJUD_API_KEY` no arquivo `.env` local ou como variável de ambiente.

## Fluxo recomendado

1. Execute a célula de configuração.
2. Revise os anos na célula de configuração.
3. Execute a coleta; os arquivos brutos e checkpoints ficam em `data/raw/datajud/`.
4. Execute a consolidação; será criado `tabela_principal_helios_2000_2024.parquet`.
5. Valide o Parquet e, se desejar, use `python manage.py importar_datajud` para testar o SQLite.
6. No servidor, o mesmo arquivo poderá ser usado para alimentar o PostgreSQL.

In [ ]:
from pathlib import Path
import json
import os

def localizar_projeto():
    configurado = os.getenv('HELIO_PROJECT_ROOT')
    if configurado:
        raiz = Path(configurado).expanduser().resolve()
        if (raiz / 'manage.py').is_file():
            return raiz
    atual = Path.cwd().resolve()
    for raiz in (atual, *atual.parents):
        if (raiz / 'manage.py').is_file():
            return raiz
    raise RuntimeError('Execute o notebook dentro do projeto Hélio ou defina HELIO_PROJECT_ROOT.')

def carregar_env(caminho):
    if not caminho.is_file():
        return
    for linha in caminho.read_text(encoding='utf-8').splitlines():
        linha = linha.strip()
        if not linha or linha.startswith('#') or '=' not in linha:
            continue
        chave, valor = linha.split('=', 1)
        chave = chave.strip().lstrip(chr(65279))
        valor = valor.strip().strip(chr(34)).strip(chr(39))
        if not os.environ.get(chave):
            os.environ[chave] = valor

PROJECT_ROOT = localizar_projeto()
carregar_env(PROJECT_ROOT / '.env')

print(f'Projeto: {PROJECT_ROOT}')
print('O notebook não acessa nenhum banco de dados durante a coleta.')

In [ ]:
import time
from datetime import datetime
import requests

API_KEY = os.getenv('DATAJUD_API_KEY', '').strip()
if not API_KEY:
    raise RuntimeError('DATAJUD_API_KEY não encontrada. Preencha o .env local; não coloque a chave no notebook.')

URL_DATAJUD = os.getenv(
    'DATAJUD_URL',
    'https://api-publica.datajud.cnj.jus.br/api_publica_tjba/_search',
)
ANOS = [int(valor) for valor in os.getenv('DATAJUD_ANOS', ','.join(str(ano) for ano in range(2000, 2025))).split(',') if valor.strip()]
LIMITE_POR_ANO = int(os.getenv('DATAJUD_LIMITE_POR_ANO', '0') or 0) or None
PASTA_RAW = PROJECT_ROOT / 'data' / 'raw' / 'datajud'
PASTA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'datajud'
ARQUIVO_PARQUET_ATUAL = PASTA_PROCESSED / 'tabela_principal_helios.parquet'
ARQUIVO_PARQUET = PASTA_PROCESSED / 'tabela_principal_helios_2000_2024.parquet'
PASTA_RAW.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

LOTE_INICIAL = 5_000
TIMEOUT = (30, 300)
CAMPOS_API = [
    'grau', 'numeroProcesso', 'orgaoJulgador.codigo',
    'orgaoJulgador.nome', 'orgaoJulgador.codigoMunicipioIBGE',
    'classe.codigo', 'classe.nome', 'assuntos.codigo', 'assuntos.nome',
]
LIMITES_MODELO = {
    'grau': 2, 'processo': 25, 'id_municipio': 7, 'municipio': 27,
    'nome_classe': 310, 'codigo_orgao': 5, 'nome_orgao': 155,
}

session = requests.Session()
session.headers.update({
    'Authorization': f'ApiKey {API_KEY}',
    'Content-Type': 'application/json',
    'Accept': 'application/json',
    'Accept-Encoding': 'gzip',
})

def caminho_ano(ano):
    pasta = PASTA_RAW / str(ano)
    pasta.mkdir(parents=True, exist_ok=True)
    return {
        'parcial': pasta / f'tjba_{ano}_raw.jsonl.part',
        'final': pasta / f'tjba_{ano}_raw.jsonl',
        'checkpoint': pasta / f'checkpoint_{ano}_raw.json',
    }

def transformar_hit(hit, ano):
    source = hit.get('_source') or {}
    orgao = source.get('orgaoJulgador') or {}
    classe = source.get('classe') or {}
    assuntos = source.get('assuntos') or []
    if isinstance(assuntos, dict):
        assuntos = [assuntos]
    codigos = [str(item.get('codigo')) for item in assuntos if isinstance(item, dict) and item.get('codigo') is not None]
    nomes = [str(item.get('nome') or '') for item in assuntos if isinstance(item, dict) and item.get('codigo') is not None]
    registro = {
        'ano': int(ano),
        'grau': str(source.get('grau') or ''),
        'processo': str(source.get('numeroProcesso') or ''),
        'id_municipio': str(orgao.get('codigoMunicipioIBGE') or ''),
        'municipio': str(source.get('municipio') or ''),
        'codigo_classe': str(classe.get('codigo') or ''),
        'nome_classe': str(classe.get('nome') or ''),
        'codigo_assunto': ' | '.join(codigos),
        'nome_assunto': ' | '.join(nomes),
        'codigo_orgao': str(orgao.get('codigo') or ''),
        'nome_orgao': str(orgao.get('nome') or ''),
    }
    for campo, limite in LIMITES_MODELO.items():
        if len(registro[campo]) > limite:
            raise ValueError(f'{campo} excede o limite do modelo: {len(registro[campo])} > {limite}')
    return registro

def filtro_ano(ano):
    return {'range': {'dataAjuizamento': {'gte': f'{ano}0101000000', 'lt': f'{ano + 1}0101000000'}}}

def consultar_pagina(ano, search_after=None, lote=LOTE_INICIAL):
    payload = {
        'size': lote, 'track_total_hits': False, '_source': CAMPOS_API,
        'query': filtro_ano(ano), 'sort': [{'id.keyword': {'order': 'asc'}}],
    }
    if search_after is not None:
        payload['search_after'] = search_after
    for tentativa in range(1, 8):
        try:
            resposta = session.post(URL_DATAJUD, json=payload, params={'allow_partial_search_results': 'false'}, timeout=TIMEOUT)
            if resposta.status_code == 200:
                dados = resposta.json()
                if dados.get('timed_out') or (dados.get('_shards') or {}).get('failed', 0):
                    raise RuntimeError('A API retornou timeout ou falha de shard.')
                hits = (dados.get('hits') or {}).get('hits')
                if not isinstance(hits, list):
                    raise RuntimeError('Resposta sem hits.hits.')
                return hits
            if resposta.status_code not in {429, 500, 502, 503, 504}:
                resposta.raise_for_status()
            raise RuntimeError(f'HTTP {resposta.status_code}')
        except (requests.RequestException, ValueError, RuntimeError) as erro:
            if tentativa == 7:
                raise
            espera = min(60, 2 ** tentativa)
            print(f'Tentativa {tentativa} falhou ({erro}); nova tentativa em {espera}s.')
            time.sleep(espera)

print(f'API configurada: {URL_DATAJUD}')
print(f'Anos configurados: {ANOS}')
print(f'Pasta dos arquivos brutos: {PASTA_RAW}')
print(f'Arquivo existente preservado: {ARQUIVO_PARQUET_ATUAL}')
print(f'Novo arquivo Parquet de saída: {ARQUIVO_PARQUET}')

In [ ]:
def salvar_checkpoint(caminho, estado):
    temporario = caminho.with_suffix('.tmp')
    temporario.write_text(json.dumps(estado, ensure_ascii=False, indent=2), encoding='utf-8')
    temporario.replace(caminho)

def coletar_ano(ano):
    caminhos = caminho_ano(ano)
    if caminhos['final'].is_file() and caminhos['final'].stat().st_size > 0:
        print(f'{ano}: arquivo final já existe; coleta não repetida.')
        return caminhos['final']
    parcial = caminhos['parcial']
    checkpoint_path = caminhos['checkpoint']
    if checkpoint_path.is_file():
        estado = json.loads(checkpoint_path.read_text(encoding='utf-8'))
    elif parcial.is_file() and parcial.stat().st_size > 0:
        raise RuntimeError(f'{ano}: existe arquivo parcial sem checkpoint; não vou sobrescrever o progresso.')
    else:
        estado = {'ano': ano, 'search_after': None, 'processos': 0, 'file_size': 0, 'concluido': False}
    if parcial.is_file() and parcial.stat().st_size > int(estado.get('file_size', 0)):
        with parcial.open('r+b') as arquivo:
            arquivo.truncate(int(estado.get('file_size', 0)))
    if estado.get('concluido') and parcial.is_file():
        parcial.replace(caminhos['final'])
        return caminhos['final']
    total = int(estado.get('processos', 0))
    search_after = estado.get('search_after')
    lote = LOTE_INICIAL
    with parcial.open('ab') as arquivo:
        while True:
            hits = consultar_pagina(ano, search_after, lote)
            if not hits:
                estado.update({'file_size': arquivo.tell(), 'concluido': True})
                salvar_checkpoint(checkpoint_path, estado)
                parcial.replace(caminhos['final'])
                print(f'{ano}: coleta concluída com {total:,} registros.')
                return caminhos['final']
            novo_search_after = hits[-1].get('sort')
            if not novo_search_after or novo_search_after == search_after:
                raise RuntimeError(f'{ano}: search_after não avançou.')
            linhas = []
            for hit in hits:
                linhas.append(json.dumps(transformar_hit(hit, ano), ensure_ascii=False, separators=(',', ':')).encode('utf-8') + b'\n')
            for linha in linhas:
                arquivo.write(linha)
            arquivo.flush()
            total += len(hits)
            search_after = novo_search_after
            estado.update({'search_after': search_after, 'processos': total, 'file_size': arquivo.tell(), 'concluido': False})
            salvar_checkpoint(checkpoint_path, estado)
            print(f'{ano}: {total:,} registros coletados')
            if LIMITE_POR_ANO and total >= LIMITE_POR_ANO:
                print(f'{ano}: limite de teste atingido; arquivo parcial preservado.')
                return parcial

# Coleta todos os anos configurados; por padrão, de 2000 a 2024.
for ano in ANOS:
    coletar_ano(ano)

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

COLUNAS_PARQUET = [
    'ano', 'grau', 'processo', 'id_municipio', 'municipio',
    'codigo_classe', 'nome_classe', 'codigo_assunto', 'nome_assunto',
    'codigo_orgao', 'nome_orgao',
]
SCHEMA_PARQUET = pa.schema([
    pa.field('ano', pa.int16()),
    *[pa.field(coluna, pa.string()) for coluna in COLUNAS_PARQUET[1:]],
])

def fontes_coletadas():
    fontes = []
    for ano in ANOS:
        caminhos = caminho_ano(ano)
        if caminhos['final'].is_file():
            fontes.append((ano, caminhos['final'], caminhos['final'].stat().st_size))
        elif caminhos['parcial'].is_file():
            if not caminhos['checkpoint'].is_file():
                raise FileNotFoundError(f'Checkpoint ausente para o ano {ano}.')
            checkpoint = json.loads(caminhos['checkpoint'].read_text(encoding='utf-8'))
            fontes.append((ano, caminhos['parcial'], int(checkpoint['file_size'])))
    if not fontes:
        raise FileNotFoundError('Nenhuma coleta encontrada. Execute coletar_ano primeiro.')
    return fontes

def iterar_registros(caminho, limite):
    with caminho.open('rb') as arquivo:
        while arquivo.tell() < limite:
            linha = arquivo.readline(limite - arquivo.tell())
            if not linha:
                break
            if linha.strip():
                yield json.loads(linha.decode('utf-8'))

def consolidar_parquet():
    fontes = fontes_coletadas()
    temporario = Path(str(ARQUIVO_PARQUET) + '.tmp')
    total = 0
    lote = []
    if temporario.exists():
        temporario.unlink()
    with pq.ParquetWriter(temporario, SCHEMA_PARQUET, compression='snappy') as escritor:
        for ano, caminho, limite in fontes:
            lidos_ano = 0
            for registro in iterar_registros(caminho, limite):
                registro = {coluna: registro.get(coluna, '') for coluna in COLUNAS_PARQUET}
                registro['ano'] = int(registro['ano'])
                lote.append(registro)
                lidos_ano += 1
                total += 1
                if len(lote) >= 10_000:
                    escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA_PARQUET))
                    lote.clear()
            print(f'{ano}: {lidos_ano:,} registros preparados para o Parquet')
        if lote:
            escritor.write_table(pa.Table.from_pylist(lote, schema=SCHEMA_PARQUET))
    if total == 0:
        temporario.unlink(missing_ok=True)
        raise RuntimeError('Nenhum registro foi consolidado.')
    validacao = pq.ParquetFile(temporario)
    linhas_validadas = validacao.metadata.num_rows
    esquema_validado = validacao.schema_arrow
    validacao.close()
    if linhas_validadas != total or esquema_validado != SCHEMA_PARQUET:
        temporario.unlink(missing_ok=True)
        raise RuntimeError('A validação do Parquet falhou; o arquivo final não foi alterado.')
    temporario.replace(ARQUIVO_PARQUET)
    print(f'Parquet salvo: {ARQUIVO_PARQUET}')
    print(f'Registros: {total:,}')
    print(f'Tamanho: {ARQUIVO_PARQUET.stat().st_size / 1024**2:,.2f} MB')
    return ARQUIVO_PARQUET

# Execute depois que a coleta dos anos desejados terminar.
consolidar_parquet()

## Carga opcional do Parquet no banco local

A população do banco não faz parte da consolidação. Depois de gerar o arquivo, use o comando do projeto apontando para o Parquet. Assim, o mesmo arquivo poderá ser validado localmente e depois enviado ao PostgreSQL.

In [ ]:
print(f'Arquivo histórico gerado: {ARQUIVO_PARQUET}')
print(f'Arquivo atual preservado: {ARQUIVO_PARQUET_ATUAL}')
print('O comando sem --arquivo usa os dois arquivos por padrão:')
print('python manage.py importar_datajud --dry-run')
print('python manage.py importar_datajud --limpar-antes')

In [ ]:
parquet = pq.ParquetFile(ARQUIVO_PARQUET)
print(f'Arquivo: {ARQUIVO_PARQUET}')
print(f'Registros: {parquet.metadata.num_rows:,}')
print(f'Grupos de linhas: {parquet.metadata.num_row_groups}')
print('Colunas:', parquet.schema_arrow.names)
amostra = parquet.read_row_group(0, columns=['ano', 'grau', 'id_municipio', 'nome_classe']).slice(0, 5).to_pylist()
print('Amostra:', amostra)

## Configuração no servidor

No servidor, mantenha o mesmo notebook e configure as variáveis de ambiente antes da execução:

```text
DATAJUD_API_KEY=...
DATAJUD_ANOS=2000,2001,...,2024
```

O notebook não imprime a chave, não contém credenciais e salva o histórico em `data/processed/datajud/tabela_principal_helios_2000_2024.parquet`. O comando `python manage.py importar_datajud` importa esse arquivo e o arquivo existente de 2025–2026.